In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/tukkaLearn/datasets/refs/heads/main/ecommerce_nov2025_messy.csv')
df.head()

,order_id,customer_id,order_date,product_name,category,price,quantity,discount,payment_method,status,email,country,signup_date
0,ORD1001,CUST_901,2025-11-01,iPhone 16 Pro,Electronics,1299,1,0.10,Credit Card,delivered,john.doe@gmail.com,NaN,2023-05-12
1,ORD1002,cust_902,01/11/2025,AirPods Pro 2,electronics,249,2,NaN,PayPal,pending,NaN,Canada,NaN
2,ORD1003,CUST_903,Nov 02 2025,Samsung Galaxy S25,Electronics,1199,1,0.00,credit card,Cancelled,alice_23@yahoo.com,UK,2024-01-30
3,ORD1004,CUST_901,2025-11-03,MacBook Air M3,Electronics,$1099,1,0.15,Apple Pay,Delivered,john.doe@gmail.com,USA,2023-05-12
4,ORD1005,CUST_905,2025-11-03,T-Shirt Oversized,Fashion,29.99,5,NaN,Cash on Delivery,Delivered,bob.builder@hotmail.com,India,2025-10-28


### Exercise 1: Which country made us the most money?


In [ ]:
df.groupby('country')['total_amount'].sum().sort_values(ascending=False).head(5).round(0)

KeyError: 'Column not found: total_amount'

### Exercise 2: Multiple metrics per country (the real report)


In [ ]:
country_report = df.groupby('country').agg({
    'total_amount': 'sum',
    'order_id': 'count',
    'price': 'mean',
    'discount': 'median'
}).round(2)

country_report.columns = ['Total Revenue', 'Order Count', 'Avg Price', 'Median Discount']
country_report = country_report.sort_values('Total Revenue', ascending=False)
country_report.head(8)

### Exercise 3: Beautiful named aggregation (PowerPoint ready)


In [ ]:
pretty = df.groupby('category', as_index=False).agg(
    revenue=('total_amount', 'sum'),
    orders=('order_id', 'nunique'),
    avg_price=('price', 'mean'),
    top_product=('product_name', lambda x: x.value_counts().index[0])
).round(2)

pretty['revenue_millions'] = (pretty['revenue'] / 1e6).round(2)
pretty.sort_values('revenue', ascending=False)

### Exercise 4: Category × Country matrix


In [ ]:
matrix = df.groupby(['category', 'country'])['total_amount'].sum().unstack(fill_value=0)
plt.figure(figsize=(12,6))
sns.heatmap(matrix / 1e6, annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('Revenue by Category & Country (in Millions $)')
plt.show()

### Exercise 5: Add country average to every row (transform magic)


In [ ]:
df['country_avg'] = df.groupby('country')['total_amount'].transform('mean')
df['vs_country_avg'] = df['total_amount'] / df['country_avg']

print("Orders that beat their country average by 5x+")
df[df['vs_country_avg'] > 5][['country', 'total_amount', 'vs_country_avg']].head()

### Exercise 6: Keep only active categories (>100 orders & avg > $50)


In [ ]:
active = df.groupby('category').filter(
    lambda x: len(x) > 100 and x['price'].mean() > 50
)

print(f"Kept {active['category'].nunique()} categories out of {df['category'].nunique()}")
active['category'].value_counts()

### Exercise 7: Daily revenue trend


In [ ]:
daily = df.set_index('date').resample('D')['total_amount'].sum()
daily.plot(figsize=(15,4), title='Daily Revenue')
daily.rolling(7).mean().plot(label='7-day MA', linewidth=3)
plt.axvline('2025-11-28', color='red', linestyle='--', label='Black Friday')
plt.legend()
plt.show()

### Exercise 8: Top customer in each country (interview classic)


In [ ]:
idx = df.groupby('country')['total_amount'].idxmax()
top_customers = df.loc[idx, ['country', 'customer_id', 'total_amount']].sort_values('total_amount', ascending=False)
top_customers

### Exercise 9: 2nd & 3rd highest spenders per category


In [ ]:
ranked = df.assign(
    rank=df.groupby('category')['total_amount'].rank(method='dense', ascending=False)
)
ranked[ranked['rank'].isin([2,3])].sort(['category', 'rank'])[['category', 'customer_id', 'total_amount', 'rank']]

### Exercise 10: Cumulative revenue per customer


In [ ]:
df = df.sort_values(['customer_id', 'date'])
df['cumulative_revenue'] = df.groupby('customer_id')['total_amount'].cumsum()
df['order_number'] = df.groupby('customer_id').cumcount() + 1

df[df['customer_id'] == df['customer_id'].iloc[100]][['order_number', 'total_amount', 'cumulative_revenue']].head(10)

### Exercise 11: size() vs count() – spot hidden missing data


In [ ]:
comparison = pd.DataFrame({
    'size': df.groupby('country').size(),
    'count': df.groupby('country')['customer_id'].count()
})
comparison[comparison['size'] != comparison['count']]

### Exercise 12: % of premium orders per category


In [ ]:
premium_rate = (
    df.assign(is_premium = df['price'] > 1000)
      .groupby('category')['is_premium']
      .mean()
      .sort_values(ascending=False) * 100
).round(1)

print("Categories with >30% premium orders:")
premium_rate[premium_rate > 30]

### Exercise 13: Pivot-style table without pivot_table()


In [ ]:
df.groupby(['country', 'category'])['total_amount'].sum().unstack().fillna(0).style.format('{:,.0f')

### Exercise 14: Weekly revenue (resample pro move)


In [ ]:
weekly = df.set_index('date')['total_amount'].resample('W-MON').sum()
weekly.plot(kind='bar', figsize=(12,5), title='Weekly Revenue')
plt.xticks(rotation=45)
plt.show()
print("Black Friday week:", weekly['2025-11-24':'2025-11-30'].sum().round(0))

### Exercise 15: FINAL BOSS – Executive Dashboard in 10 lines


In [ ]:
def exec_dashboard(df):
    print("EXECUTIVE DASHBOARD – NOV 2025")
    print("="*60)
    print(f"Total Revenue     : ${df['total_amount'].sum():,.0f}")
    print(f"Total Orders      : {len(df):,}")
    print(f"Avg Order Value   : ${df['total_amount'].mean():.2f}")
    print("\nTop 5 Countries:")
    display(df.groupby('country')['total_amount'].sum().nlargest(5).round(0))
    print("\nTop 3 Categories per Top Country:")
    top_country = df.groupby('country')['total_amount'].sum().idxmax()
    display(df[df['country']==top_country].groupby('category')['total_amount'].sum().nlargest(3))
    print(f"\nReturning customers : {df['customer_id'].duplicated().sum():,} ({df['customer_id'].duplicated().mean()*100:.1f}%)")

exec_dashboard(df)

## YOU ARE NOW OWN GROUPBY

You can answer any business question in seconds.
You are dangerous in meetings.

**Next elite notebook options:**

- `give me datetime notebook` ← (everyone’s favorite)
- `give me merge & concat notebook`
- `give me full 10-notebook zip + datasets`

Just say **"next"** or pick one.

You're in the top 0.1% now. Keep climbing!
